# Rossmann Store Sales Forecasting — End-to-End Analytical Walkthrough

**Objective:** forecast daily sales for 1,115 stores over a 42-day horizon. The notebook is the narrative entry point; reusable implementation remains in `src/`.

Run from the repository root after installing `requirements.txt` and placing the competition CSVs in `data/raw/`. Raw data is not committed. Saved results referenced below are already versioned in this repository.

## 1. Problem framing and success criteria

The task is a global store-level daily sales forecast over six weeks. Evaluation uses chronological validation and recursive multi-step inference. Compare MAE, RMSE, pooled WMAPE and RMSLE; then inspect store, volume, promotion, calendar and directional errors. Model choice balances accuracy, recursive validity, explainability and operational risk.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
ROOT = Path.cwd()
assert (ROOT / 'src').exists(), 'Run this notebook from the repository root.'
print('Project root:', ROOT)

## 2. Data audit and integrity

Training data: 1,017,209 rows, 1,115 stores, 2013-01-01 to 2015-07-31. There are 844,392 open store-days and 172,817 closed store-days; 54 open-store zero-sales rows; no negative sales; weekday/date validation found no mismatch. The audit workbooks and detailed findings are in `reports/data_audit.xlsx`, `reports/data_findings.md` and related investigation files.

In [ ]:
# Inspect the persisted audit findings and data-audit workbook inventory.
audit_path = ROOT / 'reports' / 'data_findings.md'
print(audit_path.read_text(encoding='utf-8')[:5000])
audit_book = pd.ExcelFile(ROOT / 'reports' / 'data_audit.xlsx')
print('Audit workbook sheets:', audit_book.sheet_names)

## 3. Exploratory data analysis

EDA covered daily and monthly trends, weekday patterns, sales distribution, daily Promo, Promo2, combined promotion states, customer counts, store type, assortment, holidays and competition. Examples: December mean sales 8,608.96; daily Promo observations averaged 8,228.28 versus 5,929.41 without Promo. These are descriptive associations, not causal effects. Sunday observations are sparse and should not be treated as network-representative. See `reports/eda_findings.md`, `reports/customer_eda_findings.md` and `reports/figures/`.

In [ ]:
eda = pd.read_csv(ROOT / 'reports' / 'baseline_results.csv')
display(eda)
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports' / 'figures' / '01_daily_sales_trend.png')))

## 4. Leakage assessment and data decisions

`Customers` is excluded because it is unavailable in test/future inference despite historical correlation with Sales (0.8236). `Id` is only a prediction-mapping key. Sales is not imputed; zero and extreme observations are retained. Closed stores receive zero forecasts. Future actual sales are never used to build validation/test lags or rolling statistics. Competition placeholders are flagged; meaningful negative competition ages are retained as future-opening information. StateHoliday codes follow the competition data dictionary.

## 5. Feature engineering

The configured candidate set has 34 features; constant-feature removal on training data only selects 33 for the final model. Groups: store/calendar, promotion/holiday, competition, and historical sales. Selected feature schema is frozen for inference. Historical features include lags 1/7/14/28, rolling means 7/14/28, store mean and store-weekday mean. Implementation: `src/features/build_features.py`, `src/features/recursive_forecasting.py`, `src/models/feature_config.py`.

In [ ]:
from src.models.feature_config import CANDIDATE_FEATURES, CATEGORICAL_FEATURES
print('Configured candidate features:', len(CANDIDATE_FEATURES))
print('Categorical features:', CATEGORICAL_FEATURES)
print('\n'.join(CANDIDATE_FEATURES))

## 6. Forecasting protocol and baselines

Both validation windows span 42 consecutive days and all 1,115 stores (46,830 Store-Date rows). Window 1 trains through 8 May 2015 and validates 9 May–19 June; Window 2 trains through 19 June and validates 20 June–31 July. Forecasts are recursive: predicted sales update historical state one day at a time. Baselines include lag-7 seasonal naive, 7-day moving average and lag-1 naive.

In [ ]:
baseline = pd.read_csv(ROOT / 'reports' / 'first_window_baseline_results.csv')
display(baseline)
print('Baseline implementation: src/models/baselines.py and src/models/run_first_window_baselines.py')

## 7. Candidate models and tuning

Random Forest and LightGBM were evaluated. Optuna tuned LightGBM against Window 1 recursive WMAPE. The selected parameters were frozen before Window 2; the later holdout was not used for further tuning. Source: `src/models/train_random_forest.py`, `train_lightgbm.py`, `tune_lightgbm.py`, and `evaluate_tuned_lightgbm_second_window.py`.

In [ ]:
tuning_metrics = json.loads((ROOT / 'models/experiments/lightgbm_tuning/lightgbm_tuning_metrics.json').read_text())
best_params = json.loads((ROOT / 'models/experiments/lightgbm_tuning/lightgbm_tuning_best_params.json').read_text())
print('Window 1 tuned LightGBM metrics:', tuning_metrics)
print('Frozen hyperparameters:', best_params)

## 8. Model comparison and selection

Tuned LightGBM led the evaluated models on MAE, RMSE, WMAPE and RMSLE in both windows. Window 1 is model-selection evidence; Window 2 is the later fixed-parameter holdout.

In [ ]:
comparison = pd.DataFrame([
 {'Window':'W1','Model':'Seasonal naive (lag 7)','MAE':2168.95,'RMSE':3745.19,'WMAPE %':37.2313,'RMSLE':3.6797},
 {'Window':'W1','Model':'Random Forest','MAE':511.64,'RMSE':811.90,'WMAPE %':8.7826,'RMSLE':0.1185},
 {'Window':'W1','Model':'Tuned LightGBM','MAE':455.60,'RMSE':708.49,'WMAPE %':7.8206,'RMSLE':0.1085},
 {'Window':'W2','Model':'Random Forest','MAE':554.59,'RMSE':866.16,'WMAPE %':9.2387,'RMSLE':0.1155},
 {'Window':'W2','Model':'Untuned LightGBM','MAE':525.13,'RMSE':817.43,'WMAPE %':8.7480,'RMSLE':0.1102},
 {'Window':'W2','Model':'Tuned LightGBM','MAE':514.10,'RMSE':807.69,'WMAPE %':8.5643,'RMSLE':0.1077},
])
display(comparison)

## 9. Detailed holdout evaluation and error analysis

Window 2 tuned LightGBM: all rows MAE 514.10, RMSE 807.69, WMAPE 8.5643%, RMSLE 0.1077; open rows MAE 597.67, RMSE 870.87, WMAPE 8.5643%, RMSLE 0.1161. The identical WMAPE occurs because closed rows contribute zero actual and predicted sales. Store-level results, StoreType, sales-volume, promotion and calendar analyses are persisted under `src/evaluation/results/`.

In [ ]:
holdout = json.loads((ROOT / 'reports/second_window_tuned_lightgbm_results.json').read_text())
print(json.dumps(holdout, indent=2))
store_type = pd.read_csv(ROOT / 'src/evaluation/results/storetype_level_performance.csv')
display(store_type[['StoreType','NumberOfStores','WMAPE_Pct','MeanBias','Underforecast_Rate_Pct','Overforecast_Rate_Pct']])

## 10. Explainability

Native LightGBM TreeSHAP contributions were evaluated on 40,282 open-store observations and 33 features. Additivity validation passed with maximum error 1.24e-10. The leading global contributors were store-weekday historical mean, Promo, Sales_Lag_1, Day, Sales_Lag_14 and Sales_RollingMean_28. Attribution explains model behaviour, not causality.

In [ ]:
shap = pd.read_csv(ROOT / 'src/evaluation/results/shap_global_importance.csv')
display(shap[['Feature','MeanAbsSHAP']].head(10))
validation = json.loads((ROOT / 'src/evaluation/results/shap_contribution_validation.json').read_text())
print(validation)

## 11. Business interpretation, risks and recommendation

Window 2 has positive aggregate bias: 61.51% of open-store forecasts are overforecasts; mean bias is +204.51 per open store-day and total bias is approximately +8.24m (2.93% of actual sales). Low-volume days and promotion-active periods have higher proportional error. Advance tuned LightGBM to production-readiness validation with store-level monitoring, bias review, promotion diagnostics, uncertainty estimates and shadow rollout. Do not deploy solely on aggregate WMAPE.

## 12. Reproducibility and source map

The notebook is a consolidated review layer; modular source code remains authoritative. See `README.md` for environment and data setup. Key scripts: preprocessing (`src/preprocessing/`), features (`src/features/`), models/tuning (`src/models/`), error analysis and explainability (`src/evaluation/`). The detailed report is `reports/model_evaluation_report.md`; architecture is `architecture/production_architecture.md`.